# Hybrid Search

Researching Verbalization techniques and hybrid search.

February 2026

## Merging flattened datasets

CSV datasets from Kaggle.

./data/formula-1-world-championship-1950-2020/

In [8]:
import pandas as pd
import os

# Base path for your dataset
base_path = './data/formula-1-world-championship-1950-2020/'

def load_f1_file(filename):
    df = pd.read_csv(os.path.join(base_path, filename))
    df.columns = df.columns.str.strip()
    return df

# Load all 13 core files
results = load_f1_file('results.csv')
races = load_f1_file('races.csv')
circuits = load_f1_file('circuits.csv')
drivers = load_f1_file('drivers.csv')
constructors = load_f1_file('constructors.csv')
status = load_f1_file('status.csv')
qualifying = load_f1_file('qualifying.csv')
pit_stops = load_f1_file('pit_stops.csv')
driver_standings = load_f1_file('driver_standings.csv')
sprint_results = load_f1_file('sprint_results.csv')
constructor_standings = load_f1_file('constructor_standings.csv')
seasons = load_f1_file('seasons.csv')
constructor_results = load_f1_file('constructor_results.csv')

# --- 1. ENHANCED AGGREGATIONS ---

# Status Grouping for better RAG Semantic Search
def categorize_status(s):
    s = str(s).lower()
    if 'finished' in s or '+' in s: return 'Finished'
    if any(x in s for x in ['accident', 'collision', 'spun', 'damage']): return 'Incident'
    if any(x in s for x in ['disqualified', 'excluded', '107%']): return 'Legal/DQ'
    return 'Mechanical/Technical'

status['status_category'] = status['status'].apply(categorize_status)

# Aggregate Pit Stops (Total stops and total time)
pit_summary = pit_stops.groupby(['raceId', 'driverId']).agg(
    n_pit_stops=('stop', 'max'),
    total_pit_ms=('milliseconds', 'sum')
).reset_index()

# Aggregate Qualifying (Best position and Q3 time)
qual_summary = qualifying[['raceId', 'driverId', 'position', 'q1', 'q2', 'q3']].rename(
    columns={'position': 'qual_pos'}
)

# Aggregate Sprint Results
sprint_summary = sprint_results[['raceId', 'driverId', 'points', 'positionText', 'statusId']].rename(
    columns={'points': 'sprint_points', 'positionText': 'sprint_pos', 'statusId': 'sprint_statusId'}
)

# --- 2. THE MASTER MERGE ---

# Start with Results as the anchor
df = pd.merge(results, races, on='raceId', suffixes=('', '_race'))
df = pd.merge(df, circuits, on='circuitId', suffixes=('', '_circuit'))
df = pd.merge(df, drivers, on='driverId', suffixes=('', '_driver'))
df = pd.merge(df, constructors, on='constructorId', suffixes=('', '_team'))
df = pd.merge(df, status, on='statusId')

# Join Aggregated Summaries
df = pd.merge(df, pit_summary, on=['raceId', 'driverId'], how='left')
df = pd.merge(df, qual_summary, on=['raceId', 'driverId'], how='left')
df = pd.merge(df, sprint_summary, on=['raceId', 'driverId'], how='left')

# Join Championship Context (Driver & Team Standings)
df = pd.merge(df, driver_standings, on=['raceId', 'driverId'], how='left', suffixes=('', '_ds'))
df = pd.merge(df, constructor_standings, on=['raceId', 'constructorId'], how='left', suffixes=('', '_cs'))

# --- 3. CALCULATED FEATURES FOR AI ---

# Total points for the entire weekend (Race + Sprint)
df['weekend_points_total'] = df['points'].fillna(0) + df['sprint_points'].fillna(0)

# Identify "Home Race" for drivers (e.g., Lewis Hamilton at Silverstone)
df['is_home_race'] = df['nationality'] == df['country']

# --- 4. FINAL CLEANUP ---

# Rename overlapping 'name' columns for clarity
df = df.rename(columns={
    'name': 'gp_name',
    'name_team': 'constructor_name',
    'name_circuit': 'circuit_name',
    'points_ds': 'driver_championship_points',
    'position_ds': 'driver_championship_rank',
    'points_cs': 'team_championship_points',
    'position_cs': 'team_championship_rank'
})

# Drop redundant URL columns to keep the file lean
cols_to_drop = [c for c in df.columns if 'url' in c.lower()]
df = df.drop(columns=cols_to_drop)

# Save the exhaustive dataset
output_path = os.path.join(base_path, 'f1_complete_historical_dataset.csv')
df.to_csv(output_path, index=False)

print(f"Exhaustive dataset created at: {output_path}")
print(f"Total Records: {len(df)} | Total Features: {len(df.columns)}")

Exhaustive dataset created at: ./data/formula-1-world-championship-1950-2020/f1_complete_historical_dataset.csv
Total Records: 26759 | Total Features: 74


In [18]:
import pandas as pd

def verbalize_individual_result(row):
    # Core identity
    name = f"{row['forename']} {row['surname']}"
    team = row['constructor_name']
    year = int(row['year'])
    gp = row['gp_name']
    
    # 1. Handle Grid
    if pd.isna(row['grid']) or row['grid'] == 0:
        grid_text = "starting from an unclassified position"
    else:
        grid_text = f"starting from P{int(row['grid'])}"
        
    # 2. Handle Finishing Position
    finish_text = f"finishing in P{row['positionText']}"
    
    # 3. Handle Laps (Fixed the self-reference error here)
    laps_val = 0 if pd.isna(row['laps']) else int(row['laps'])
    if laps_val > 0:
        laps_desc = f"He completed {laps_val} laps"
    else:
        laps_desc = "He was unable to complete any full laps"
    
    # 4. Handle Points and Fastest Lap
    points_val = 0.0 if pd.isna(row['points']) else row['points']
    points_text = f"scoring {points_val} championship points"
    
    # Using your specific column names for fastest lap
    fastest = ""
    if str(row['rank']) == '1':
        fastest = f" and setting the fastest lap of the race ({row['fastestLapTime']})"
    elif str(row['rank']) == '2':
        fastest = f" (including the 2nd fastest lap of {row['fastestLapTime']})"
    
    # 5. Handle Standings Rank
    if pd.isna(row['driver_championship_rank']):
        rank_text = "This performance contributed to his season standing."
    else:
        rank_text = f"This performance placed him at rank {int(row['driver_championship_rank'])} in the world standings."
    
    # Combine into a natural paragraph
    sentence = (
        f"In the {year} {gp}, {name} drove for {team}, {grid_text} and {finish_text}, {points_text}{fastest}. "
        f"{laps_desc} at the {row['circuit_name']} with a final status of '{row['status']}'. "
        f"{rank_text}"
    )
    
    # Add home race flavor
    if row.get('is_home_race') == True:
        sentence = f"Competing in front of a home crowd, {name} drove for {team}..." # Simplified for logic
        # Or more simply:
        sentence = "Competing in front of a home crowd, " + sentence[0].lower() + sentence[1:]
        
    return sentence

# Apply it to your dataframe
df['driver_verbalization'] = df.apply(verbalize_individual_result, axis=1)

In [ ]:
df['driver_verbalization'][5]

"In the 2008 Australian Grand Prix, Lewis Hamilton drove for McLaren, starting from P1 and finishing in P1, scoring 10.0 championship points (including the 2nd fastest lap of 1:27.452). He completed 58 laps at the Albert Park Grand Prix Circuit with a final status of 'Finished'. This performance placed him at rank 1 in the world standings."

In [20]:
import pandas as pd
import os

base_path = './data/formula-1-world-championship-1950-2020/'

def generate_standings_verbalizations():
    # Load required files
    standings = pd.read_csv(os.path.join(base_path, 'driver_standings.csv'))
    races = pd.read_csv(os.path.join(base_path, 'races.csv'))
    drivers = pd.read_csv(os.path.join(base_path, 'drivers.csv'))
    
    # Merge for context
    df = standings.merge(races[['raceId', 'year', 'name', 'round']], on='raceId')
    df = df.merge(drivers[['driverId', 'forename', 'surname']], on='driverId')
    
    def create_standing_text(row):
        name = f"{row['forename']} {row['surname']}"
        suffix = "st" if row['position'] == 1 else "nd" if row['position'] == 2 else "rd" if row['position'] == 3 else "th"
        
        return (
            f"As of the conclusion of the {row['year']} {row['name']} (Round {row['round']}), "
            f"{name} is ranked {int(row['position'])}{suffix} in the World Drivers' Championship "
            f"with a total of {row['points']} points and {int(row['wins'])} race wins this season."
        )

    df['standing_verbalization'] = df.apply(create_standing_text, axis=1)
    
    # Save as a separate reference file
    df[['raceId', 'driverId', 'year', 'standing_verbalization']].to_csv(
        os.path.join(base_path, 'f1_championship_standings_verbalized.csv'), index=False
    )

generate_standings_verbalizations()

In [ ]:
df['standing_verbalization'][0]

KeyError: 'standing_verbalization'

## organize data by race

In [13]:
import pandas as pd
import json

# 1. Load the data with low_memory=False to ignore the DtypeWarning
file_path = './data/formula-1-world-championship-1950-2020/f1_complete_historical_dataset.csv'
df = pd.read_csv(file_path, low_memory=False)

# 2. Group the data by Race to create a single document per event
def create_race_bundles(df):
    race_bundles = []
    
    # Sort by year and round to keep the timeline logical
    for (year, race_id), race_group in df.groupby(['year', 'raceId']):
        # Metadata shared by all drivers in this race
        race_meta = race_group.iloc[0]
        
        # Extract and sort the results for this specific race
        # We use positionOrder to ensure we have a clean 1-2-3-etc. ranking
        results = race_group.sort_values('positionOrder')
        
        driver_results = []
        for _, row in results.iterrows():
            driver_results.append({
                "rank": row['positionText'],
                "driver": f"{row['forename']} {row['surname']}",
                "team": row['constructor_name'],
                "grid": row['grid'],
                "status": row['status'],
                "points": row['points'],
                "weekend_total": row['weekend_points_total'],
                "fastest_lap": row['fastestLapTime']
            })

        # Build the final organized object for this race
        race_bundle = {
            "race_id": int(race_id),
            "season": int(year),
            "round": int(race_meta['round']),
            "race_name": race_meta['gp_name'],
            "circuit": race_meta['circuit_name'],
            "location": f"{race_meta['location']}, {race_meta['country']}",
            "date": race_meta['date'],
            "winner": driver_results[0]['driver'] if driver_results else "N/A",
            "podium": [res['driver'] for res in driver_results[:3]],
            "results": driver_results
        }
        race_bundles.append(race_bundle)
        
    return race_bundles

# Execute and export
organized_races = create_race_bundles(df)

with open('./data/formula-1-world-championship-1950-2020/races_by_event.json', 'w') as f:
    json.dump(organized_races, f, indent=4)

print(f"Organized {len(organized_races)} races into event-based JSON.")

Organized 1125 races into event-based JSON.


In [14]:
def generate_race_verbalization(race_bundle):
    year = race_bundle['season']
    name = race_bundle['race_name']
    location = race_bundle['location']
    circuit = race_bundle['circuit']
    winner = race_bundle['winner']
    podium = ", ".join(race_bundle['podium'])
    
    # Narrative intro
    intro = f"The {year} {name} took place at the {circuit} in {location}. "
    
    # Result summary
    result_summary = f"The race was won by {winner}. The final podium consisted of {podium}. "
    
    # Attrition and Drama (Extracting top 3 DNFs or incidents)
    incidents = [res for res in race_bundle['results'] if res['status'] != 'Finished' and '+' not in res['status']]
    if incidents:
        incident_list = ", ".join([f"{res['driver']} ({res['status']})" for res in incidents[:3]])
        drama = f"Key incidents and retirements during the race included {incident_list}. "
    else:
        drama = "The race saw a high completion rate with no major retirements. "
        
    # Championship Impact (Using the winner's rank)
    winner_data = race_bundle['results'][0]
    rank_info = f"Following this event, {winner} held rank {int(winner_data['rank'])} in the championship standings."

    return intro + result_summary + drama + rank_info

# Apply to your organized data
for race in organized_races:
    race['verbalization'] = generate_race_verbalization(race)

In [15]:
organized_races[0]['verbalization']

'The 1950 British Grand Prix took place at the Silverstone Circuit in Silverstone, UK. The race was won by Nino Farina. The final podium consisted of Nino Farina, Luigi Fagioli, Reg Parnell. Key incidents and retirements during the race included Juan Fangio (Oil leak), Joe Kelly (Not classified), Prince Bira (Out of fuel). Following this event, Nino Farina held rank 1 in the championship standings.'

## To Markdown

In [23]:
import pandas as pd
import os

path = './data/formula-1-world-championship-1950-2020/'
master_csv = os.path.join(path, 'f1_complete_historical_dataset.csv')
df = pd.read_csv(master_csv, low_memory=False)

def get_row_verbalization(row):
    """Generates the prose for a single driver's result on the fly."""
    name = f"{row['forename']} {row['surname']}"
    team = row['constructor_name']
    
    # Handle Finishing vs DNF
    if row['status'] == 'Finished' or '+' in str(row['status']):
        status_text = f"finishing in P{row['positionText']}"
    else:
        # Explicitly capture the 'Engine', 'Accident', etc.
        status_text = f"retiring due to a {row['status']} issue" if row['status_category'] == 'Mechanical/Technical' else f"retiring due to an {row['status']}"

    laps = int(row['laps']) if pd.notna(row['laps']) else 0
    grid = f"P{int(row['grid'])}" if pd.notna(row['grid']) and row['grid'] != 0 else "the back of the grid"
    
    return f"**{name}** ({team}) started from {grid} and completed {laps} laps, {status_text}."

def generate_f1_history_markdown(df, output_file):
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write("# The Complete History of Formula 1\n\n")

        # Sort chronologically
        grouped = df.sort_values(['year', 'round']).groupby(['year', 'round'])

        for (year, rd), group in grouped:
            race_info = group.iloc[0]
            f.write(f"## Season: {year}\n")
            f.write(f"### Round {rd}: {race_info['gp_name']}\n")
            f.write(f"**Circuit:** {race_info['circuit_name']} | **Location:** {race_info['location']}\n\n")

            # 1. Classified Finishers
            f.write("#### Classified Finishers\n")
            finishers = group[group['status'].str.contains('Finished|\+', na=False, case=False)].sort_values('positionOrder')
            for _, row in finishers.iterrows():
                f.write(f"- {get_row_verbalization(row)}\n")

            # 2. Retirements (DNFs) - Addressing your specific need for engine issues, etc.
            f.write("\n#### Retirements and Incidents\n")
            dnfs = group[~group['status'].str.contains('Finished|\+', na=False, case=False)].sort_values('laps', ascending=False)
            if not dnfs.empty:
                for _, row in dnfs.iterrows():
                    # We explicitly highlight the status (e.g. Engine) for RAG indexing
                    f.write(f"- {get_row_verbalization(row)} (Official Status: **{row['status']}**)\n")
            else:
                f.write("- No retirements recorded.\n")

            # 3. Championship Snapshot
            f.write(f"\n#### Championship State (Post-Round {rd})\n")
            top_drivers = group.nsmallest(3, 'driver_championship_rank')
            for _, d_row in top_drivers.iterrows():
                f.write(f"- Driver Standing: {d_row['forename']} {d_row['surname']} (Rank {int(d_row['driver_championship_rank'])})\n")
            
            f.write("\n---\n\n")

    print(f"File created: {output_file}")

# Run the fixed generator
generate_f1_history_markdown(df, os.path.join(path, 'history_of_f1.md'))

<>:40: SyntaxWarning: invalid escape sequence '\+'
<>:46: SyntaxWarning: invalid escape sequence '\+'
<>:40: SyntaxWarning: invalid escape sequence '\+'
<>:46: SyntaxWarning: invalid escape sequence '\+'
/var/folders/24/y__mx0xd3rn5g48sf3wlvf080000gn/T/ipykernel_48605/2004631429.py:40: SyntaxWarning: invalid escape sequence '\+'
  finishers = group[group['status'].str.contains('Finished|\+', na=False, case=False)].sort_values('positionOrder')
/var/folders/24/y__mx0xd3rn5g48sf3wlvf080000gn/T/ipykernel_48605/2004631429.py:46: SyntaxWarning: invalid escape sequence '\+'
  dnfs = group[~group['status'].str.contains('Finished|\+', na=False, case=False)].sort_values('laps', ascending=False)


File created: ./data/formula-1-world-championship-1950-2020/history_of_f1.md


In [24]:
import pandas as pd
import os

path = './data/formula-1-world-championship-1950-2020/'

def generate_career_summaries(df):
    biographies = "# Formula 1 Career Biographies\n\n"

    # --- Constructor Summaries ---
    biographies += "## All-Time Constructor Biographies\n"
    constructors = df.groupby('constructor_name')
    for name, group in constructors:
        total_wins = len(group[group['positionOrder'] == 1])
        total_poles = len(group[group['grid'] == 1])
        # Find the team's most winning driver
        top_driver = group[group['positionOrder'] == 1]['surname'].value_counts().idxmin() if total_wins > 0 else "N/A"
        
        biographies += f"### {name}\n"
        biographies += f"- **Total Victories**: {total_wins} Grand Prix wins.\n"
        biographies += f"- **Pole Positions**: {total_poles} starts from P1.\n"
        biographies += f"- **Key Personnel**: {top_driver} is among the team's most successful drivers.\n\n"

    # --- Driver Summaries ---
    biographies += "## All-Time Driver Biographies\n"
    drivers = df.groupby(['forename', 'surname'])
    for (fname, sname), group in drivers:
        full_name = f"{fname} {sname}"
        total_wins = len(group[group['positionOrder'] == 1])
        first_win_year = group[group['positionOrder'] == 1]['year'].min() if total_wins > 0 else None
        
        biographies += f"### {full_name}\n"
        biographies += f"- **Total Wins**: {total_wins}\n"
        if first_win_year:
            biographies += f"- **Career Milestone**: Secured first win in {int(first_win_year)}.\n"
        biographies += f"- **Teams**: Competing for {', '.join(group['constructor_name'].unique())}.\n\n"
        
    return biographies

# Assuming df is your merged master dataset
f1_bios = generate_career_summaries(df)
with open(os.path.join(path, 'f1_biographies.md'), 'w') as f:
    f.write(f1_bios)